### RNN（Recurrent Neural Network，循环神经网络）
RNN是深度学习领域专门用于处理“序列数据（Sequence Data）”的神经网络架构
不同于CNN处理独立同分布（i.i.d.）的数据，RNN处理“前后顺序”的数据，例如：股票价格（时间序列）、文本（单词顺序）、语音（帧顺序）、视频（帧序列）。RNN网络拥有内部状态（记忆），当前的输出不仅取决于当前的输入，还取决于对过去输入的‘记忆’。

**核心思想**
- **隐状态**：RNN神经元不仅接收当前时刻的输入$x_t$，还接受上一时刻隐藏层的状态$h_{t-1}$，能够记忆隐藏状态，存储历史记忆
- **共享参数**：RNN在时间序列的每一步，使用的权重矩阵W和U是共享的，即无论序列长度是多少，RNN都使用同一套方法处理每一步。而且所有时间共享同一组权重，大幅减少了参数量

  

### 模型原理

RNN的关键特点是隐藏状态的循环传递，即当前时刻的输出不仅依赖于当前输入，还依赖于之前所有时刻的信息，这种机制使RNN能够建模序列的时序依赖性。

隐含层神经元：  

<img src="./pictures/hidden.png" alt="隐含层神经元结构示意图" width="400" align="center" />

**RNN 前向传播**

对于时间步 \( t \)，循环神经网络（RNN）的隐藏状态和输出计算如下：

$$
h_t = f(W_x x_t + W_h h_{t-1} + b_h)
$$

$$
y_t = g(W_y h_t + b_y)
$$

其中：

- \( h_t \)：当前时刻的隐含状态  
- \( x_t \)：当前时刻的输入  
- \( y_t \)：当前时刻的输出  
- \( W_x, W_h, W_y \)：分别对应输入、隐藏状态和输出的权重矩阵  
- \( f, g \)：激活函数（例如 \( f \) 通常为 tanh 或 ReLU，\( g \) 为 softmax 或 sigmoid）

RNN在时间步上展开后，可视为多个共享参数的重复模块链式连接。序列结构过程如图所示：

<img src="./pictures/RNN.png" width="1000" align="center" />
  


### 随时间反向传播（BPTT）
RNN 的训练依然使用梯度下降，但计算梯度时，因为时间步之间相互依赖，必须使用 BPTT
要计算当前时刻 \( t \) 的损失 \( L_t \) 对权重 \( W_{hh} \) 的梯度，**不能只考虑当前步**，而必须沿着时间轴一直追溯回去，
因为 \( W_{hh} \) 在 \( t-1, t-2, \dots, 0 \) 时刻都参与了计算。

这导致了一个非常关键的数学现象：**连乘效应**。
$$
\frac{\partial L_t}{\partial W} \propto \prod_{i=k}^{t} \frac{\partial h_i}{\partial h_{i-1}}
$$

由此引发的后果：

- **梯度爆炸**：如果某个时刻的梯度值 \( > 1 \)，连乘后梯度会指数级增长。
- **梯度消失**：如果某个时刻的梯度值 \( < 1 \)，连乘后梯度会指数级衰减。

这正是 **RNN 最大的痛点**，也是后续 LSTM、GRU 等结构试图解决的核心问题。

**CNN应用场景**
- **自然语言处理**：语言模型（如 GPT 早期基于 RNN）、文本生成、机器翻译、情感分析
- **语音处理**：语音识别（如结合 CTC 损失函数）、语音合成
- **时间序列分析**：股票价格预测、传感器数据异常检测、天气预测
- **视频处理**：视频内容理解（如动作识别，结合 CNN 提取空间特征）

### RNN 的常见变体

| 变体 | 核心机制 | 解决的问题 | 现状 |
|------|----------|------------|------|
| **LSTM** | 遗忘门、输入门、输出门（门控机制，控制信息的丢弃与保留） | 通过门控让梯度能在长序列中顺畅流动，解决了**长距离依赖（梯度消失）** | 工业界**绝对主流**（文本生成、机器翻译） |
| **GRU** | 更新门、重置门（简化版的 LSTM） | 在保持 LSTM 效果的同时，参数更少，训练更快 | 在数据量不大时，GRU 效果往往优于 LSTM |
| **双向 RNN (Bi-RNN)** | 不仅有“过去”到“未来”的路径，还有“未来”到“过去”的路径 | 捕捉上下文双向信息（如预测一句话中的词） | 极其常用，在 NLP 中几乎必加 |

> 这三种变体各有侧重，实际应用时可结合任务需求和数据规模进行选择。

In [1]:
import matplotlib
matplotlib.use('TkAgg')
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']  # 'SimHei' 是黑体，也可设置 'Microsoft YaHei' 等
plt.rcParams['axes.unicode_minus'] = False  # 正确显示负号


# 设置随机种子以确保结果可复现
torch.manual_seed(42)
np.random.seed(42)


# 生成示例时间序列数据
def generate_data(n_samples=1000, seq_length=20):
    """生成简单的正弦波时间序列数据"""
    x = np.linspace(0, 10 * np.pi, n_samples + seq_length)
    y = np.sin(x)

    # 创建序列和目标
    sequences = []
    targets = []
    for i in range(n_samples):
        sequences.append(y[i:i + seq_length])
        targets.append(y[i + seq_length])

    # 转换为PyTorch张量
    sequences = torch.FloatTensor(sequences).unsqueeze(2)  # [样本数, 序列长度, 特征数]
    targets = torch.FloatTensor(targets).unsqueeze(1)  # [样本数, 1]

    # 分割训练集和测试集
    train_size = int(0.8 * n_samples)
    train_data = TensorDataset(sequences[:train_size], targets[:train_size])
    test_data = TensorDataset(sequences[train_size:], targets[train_size:])

    return train_data, test_data


# 定义RNN模型
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN层
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # 全连接输出层
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # 前向传播RNN
        out, _ = self.rnn(x, h0)

        # 我们只需要最后一个时间步的输出
        out = self.fc(out[:, -1, :])
        return out


# 训练函数
def train_model(model, train_loader, criterion, optimizer, device, epochs=100):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            # 清零梯度
            optimizer.zero_grad()

            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            # 反向传播和优化
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{epochs}], Loss: {total_loss / len(train_loader):.4f}')


# 评估函数
def evaluate_model(model, test_loader, device):
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)

            predictions.extend(outputs.cpu().numpy())
            actuals.extend(targets.cpu().numpy())

    return np.array(predictions), np.array(actuals)


# 主函数
def main():
    # 设备配置
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 超参数
    input_size = 1  # 输入特征维度
    hidden_size = 64  # 隐藏层维度
    num_layers = 1  # RNN层数
    output_size = 1  # 输出维度
    seq_length = 20  # 序列长度
    batch_size = 32  # 批次大小
    learning_rate = 0.001  # 学习率

    # 生成数据
    train_data, test_data = generate_data(seq_length=seq_length)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size)

    # 初始化模型
    model = SimpleRNN(input_size, hidden_size, num_layers, output_size).to(device)

    # 定义损失函数和优化器
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # 训练模型
    print("开始训练模型...")
    train_model(model, train_loader, criterion, optimizer, device)

    # 评估模型
    print("评估模型...")
    predictions, actuals = evaluate_model(model, test_loader, device)

    # 可视化结果
    plt.figure(figsize=(10, 6))
    plt.plot(actuals[:50], label='实际值')
    plt.plot(predictions[:50], label='预测值')
    plt.title('RNN模型预测结果')
    plt.xlabel('样本')
    plt.ylabel('值')
    plt.legend()
    plt.show()


if __name__ == "__main__":
    main()





C:\Users\31983\AppData\Local\Temp\ipykernel_27396\1273124507.py:35: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  sequences = torch.FloatTensor(sequences).unsqueeze(2)  # [样本数, 序列长度, 特征数]


开始训练模型...
Epoch [10/100], Loss: 0.0003
Epoch [20/100], Loss: 0.0001
Epoch [30/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [50/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [70/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [90/100], Loss: 0.0000
Epoch [100/100], Loss: 0.0000
评估模型...
